# Dive.ai 진행사항

> 벡터DB.ipynb / 파이프라인.ipynb 기준으로 작성된 개발 진행 현황 및 주간 일정입니다.


## 지난주 작업 (~ 2026-04-27)

### 핵심 기능 1 — 시나리오 & 캐릭터 빌더

| 항목 | 파일 | 상태 |
|------|------|------|
| ChromaDB 벡터 DB 구축 | 벡터DB.ipynb §1~5 | ✅ 완료 |
| 기승전결 스테이지 매핑 | 벡터DB.ipynb §6 | ✅ 완료 |
| RAG 검색 + 프롬프트 빌더 | 벡터DB.ipynb §7~8 | ✅ 완료 |
| 시나리오 생성 함수 | 벡터DB.ipynb §10 Cell 28 | ✅ 완료 |
| 캐릭터 생성 함수 | 벡터DB.ipynb §12 Cell 43 | ✅ 완료 |
| 빌더 통합 파이프라인 | 벡터DB.ipynb §12 Cell 44 | ✅ 완료 |

- 시나리오 생성 모델: gpt-4o-mini(전체적인 테스팅 시), gpt-4o(복잡한 프롬프트 테스팅, 시연 시), 다른 모델도 고려 후 추가
- 캐릭터 생성: AI캐릭터 + 유저캐릭터 + 주조연 2~4명 (시나리오 기반 자동 설계)

### 핵심 기능 2 — 시나리오 트랜스포머

| 항목 | 파일 | 상태 |
|------|------|------|
| 시나리오 트랜스포머 v1 (ScenarioTree) | 벡터DB.ipynb §11 Cell 30~32 | ✅ 완료 |
| 부분 재생성 함수 | 벡터DB.ipynb §11 Cell 35 | ✅ 완료 |
| GameState (호감도·선택이력·이벤트플래그) | 벡터DB.ipynb §11 Cell 37 | ✅ 완료 |
| 엔딩 조건 설계 (LLM 자동 생성) | 벡터DB.ipynb §11 Cell 38 | ✅ 완료 |
| 엔딩 판정 로직 (evaluate_ending) | 벡터DB.ipynb §11 Cell 39 | ✅ 완료 |

- 분기 구조: 기 → 승 → 전(공통) → [선택 A/B] → 전_A/전_B → 결_A/결_B - 수정 예정
- 이탈 감지 시 현재 노드 이후만 부분 재생성 (대화 요약·관계 상태 반영)


## 이번주 일정 (2026-04-28 ~ 2026-05-03)

> 목표: **웹사이트에 ui, 기능 구현, 인터랙티브 챗 시스템 프로토타입 완성**<br>
> 시나리오 트랜스포머 먼저 수정


### 트랙 1 — 시나리오 트랜스포머 수정

> 방향성 재설정 후 수정 예정

현재 구현된 구조 및 기능은 완료 상태이나,
분기 구조·이탈 감지 기준·재생성 범위 등의 **방향성을 먼저 재검토**한 뒤 수정 진행.

| 항목 | 현재 상태 | 이번주 |
|------|----------|--------|
| ScenarioTree 분기 구조 (기→승→전→[A/B]→결) | ✅ 구현 완료 | 방향성 재설정 후 수정 예정 |
| 부분 재생성 함수 (`regenerate_from_node`) | ✅ 구현 완료 | 방향성 재설정 후 수정 예정 |
| 엔딩 조건 설계 (`GameState` + 호감도/플래그) | ✅ 구현 완료 | 방향성 재설정 후 수정 예정 |


### 트랙 2 — 인터랙티브 챗 시스템 프로토타입

> 참고 파일: `벡터DB.ipynb`, `파이프라인.ipynb`

파이프라인.ipynb 로드맵 기준 **1단계 마무리 + 3단계 핵심 구현** 목표.

#### 1단계 마무리 — 로어북 자동 초기화

- 시나리오 생성 완료 시 LLM으로 핵심 항목(지명 / 인물 관계 / 고유 명사 / 복선 디테일) 자동 추출
- 추출 항목 임베딩 → ChromaDB `lorebook` 컬렉션 색인
- `generate_scenario_full()` 완료 직후 자동 호출되도록 연동

```
시나리오 생성 완료
    → LLM 핵심 항목 추출
    → 임베딩 → ChromaDB lorebook 컬렉션 색인
```

#### 2단계 마무리 — `generate_ending_conditions` 검증

- 실제 API 호출 테스트 및 `ending_condition` JSON 구조 검증
- `evaluate_ending` 통합 테스트 (해피·배드·트루 엔딩 전 케이스)
- LLM 출력 파싱 실패 시 fallback 처리 추가

#### 3단계 — 챗 시스템 핵심 구현

**시스템 프롬프트 조합 함수** (`build_system_prompt`)

| 레이어 | 내용 |
|--------|------|
| 고정 레이어 | 유저 페르소나 + 유저 노트 + 현재 ScenarioTree 노드 정보 |
| 동적 레이어 A | 로어북 Semantic Search → 문맥 유사 항목 실시간 주입 |
| 동적 레이어 B | 현재 기승전결 단계 기반 RAG 씬 컨텍스트 |
| 동적 레이어 C | 요약 기억 (10~15턴 초과 시 자동 요약 + 관계 상태 추출) |

**이탈 감지 + 분기점 처리**

| 항목 | 내용 |
|------|------|
| 이탈 감지 | N턴 연속 시나리오 방향 이탈 감지 → `regenerate_from_node()` 트리거 |
| 분기점 도달 | 현재 노드 `전` 단계 진입 시 선택지 제시 |
| 선택 기록 | `GameState.record_choice()` + `affinity_delta` 반영 |
| 엔딩 체크 | 매 턴 `check_ending_reached()` 호출 |

```
매 턴 종료 시 체크
    ├─► 분기점 도달? → 선택지 제시
    ├─► 이탈 감지?   → regenerate_from_node() 호출
    └─► 엔딩 도달?   → check_ending_reached() → 엔딩 화면
```


### 트랙 3 — 웹사이트 UI & 기능 구현 (화면 1~5)

> 참고 파일: `ui기능구현.ipynb`
> 이번주 범위: 화면 1 ~ 화면 5

#### 화면 1 — 진입: 콘텐츠 유형 & 장르 선택

- 콘텐츠 유형 선택 버튼: 만화 / 시리즈 / 영화 / 소설 / 고전
- 유형별 장르 목록 표시 + **무작위** 버튼
- 고전 선택 시 추가 UI: 국가 선택(한·중·일) → 해당 국가 장르 목록

#### 화면 2 — 소재 & 캐릭터 입력

- **섹션 A**: 소재 직접 입력 / AI에게 맡김 토글 + 자동 생성 소재 미리보기
- **섹션 B (AI 캐릭터)**: 이름·성격·외형·배경 각 입력창 우측 `AI` 뱃지 버튼
- **섹션 C (유저 캐릭터)**: 이름·성격·배경 각 입력창 우측 `AI` 뱃지 버튼
- 섹션 B/C 상단 안내 문구: `"AI 뱃지를 누르면 시나리오에 맞게 AI가 자동으로 설정해요."`
- 섹션별 **전체 AI에게 맡김** 버튼 제공
- 섹션 C 입력값 → 화면 5 페르소나 자동 연동

#### 화면 3 — 생성 중 로딩

- 단계별 진행 상태 텍스트 + 로딩 애니메이션
- 순서: 소재 분석 → 시나리오 작성 → 캐릭터 설계 → 인터랙티브 구조 변환 → 엔딩 조건 설계 → 완료

#### 화면 4 — 완료 화면: 시나리오 & 캐릭터 확인

- **섹션 A**: 기승전결 시나리오 요약 표시 (펼치기/접기, 단계별 탭)
- **섹션 B**: 등장인물 카드 (AI캐릭터 / 유저캐릭터 / 조연 2~4명) + 인라인 수정 가능
- AI 캐릭터 기준 이미지 생성 (이미지 API 미정 — 화면 구조만 구현)

#### 화면 5 — 플레이 시작 전 설정

- **유저 페르소나**: 화면 2 유저 캐릭터 정보 자동 채워짐, 자유 수정 가능
- **유저 노트**: AI가 항상 기억할 사항 자유 입력 (매 턴 시스템 프롬프트 고정 주입)
- **세션 옵션**: 출력 모델 / 추론 양 / 감성 / AI 주도 사건 / 사칭 설정 / 시작 설정
- **"대화 시작"** 버튼 → 화면 6 진입


## 전체 남은 작업 현황

| 단계 | 항목 | 상태 |
|------|------|------|
| 트랜스포머 | 분기 구조 · 이탈 감지 · 재생성 범위 방향성 재설정 | 🔲 이번주 |
| 1단계 빌더 마무리 | 로어북 자동 초기화 | 🔲 이번주 |
| 2단계 트랜스포머 마무리 | `generate_ending_conditions` 검증 | 🔲 이번주 |
| 3단계 챗 시스템 | 시스템 프롬프트 설계 (고정 + 동적 레이어 A/B/C) | 🔲 이번주 |
| 3단계 챗 시스템 | 대화 요약기 + 관계 상태 추적 | 🔲 이번주 |
| 3단계 챗 시스템 | 이탈 감지 + 분기점 도달 처리 | 🔲 이번주 |
| UI 화면 1~5 | 웹사이트 UI & 기능 구현 | 🔲 이번주 |
| 4단계 엔딩 | 엔딩 씬 생성 (LLM) | ⏳ 다음주 |
| 4단계 엔딩 | 엔딩 이미지 생성 | ⏳ API 미정 |
| 5단계 로어북 | 대화 중 실시간 참조 + 유저 관리 UI | ⏳ 다음주 |
| UI 화면 6~7 | 인터랙티브 챗 화면 + 엔딩 결과 화면 | ⏳ 다음주 |
| 6단계 세션 | 구글 로그인 + 다중 세션 + 엔딩 아카이브 | ⏳ 추후 |
| 이미지 | AI 캐릭터 이미지 생성 + 얼굴 일관성 유지 | ⏳ API 미정 |
